In [68]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [69]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [70]:
from langchain_community.document_loaders import PyPDFLoader


In [71]:
file_path = "../Data/REST-API.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
data

[Document(metadata={'producer': 'Skia/PDF m119', 'creator': 'Chromium', 'creationdate': '2025-01-21T16:18:01+00:00', 'moddate': '2025-01-21T16:18:01+00:00', 'source': '../Data/REST-API.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Notes\n1\nNotes\nUnderstanding REST APIs\nREST \ue081Representational State Transfer) is an architectural style for designing \nnetworked applications. It relies on stateless, client-server communication over \nHTTP using standard methods and status codes. RESTful APIs are designed \naround resources, which can be anything from users and products to \ndocuments.\nKey Concepts:\n\ue072\ue094\x00 Resources: Everything that can be accessed via a RESTful API is \nconsidered a "resource." Each resource has a unique identifier \ue081URI\ue082.\n\ue073\ue094\x00 Representations: Resources are transferred in some representation like \nJSON or XML.\n\ue074\ue094\x00 Stateless Communication: Each request from client to server contains all \nneede

In [72]:
question_gen = ""

for page in data:
    question_gen += page.page_content


In [73]:
#extract the page content only 
question_gen

'Notes\n1\nNotes\nUnderstanding REST APIs\nREST \ue081Representational State Transfer) is an architectural style for designing \nnetworked applications. It relies on stateless, client-server communication over \nHTTP using standard methods and status codes. RESTful APIs are designed \naround resources, which can be anything from users and products to \ndocuments.\nKey Concepts:\n\ue072\ue094\x00 Resources: Everything that can be accessed via a RESTful API is \nconsidered a "resource." Each resource has a unique identifier \ue081URI\ue082.\n\ue073\ue094\x00 Representations: Resources are transferred in some representation like \nJSON or XML.\n\ue074\ue094\x00 Stateless Communication: Each request from client to server contains all \nneeded information; the server does not store any state about the client \nsession.\n\ue075\ue094\x00 HTTP Methods: REST APIs use standard HTTP methods to perform actions \non resources.\n\ue076\ue094\x00 HTTP Status Codes: Servers use HTTP status codes to i

In [74]:
from langchain_text_splitters import TokenTextSplitter

split_ques_gen = TokenTextSplitter(

    encoding_name='cl100k_base',
    chunk_size = 1000,
    chunk_overlap = 200
)

In [75]:
chunk_ques_gen = split_ques_gen.split_text(question_gen)

In [76]:
from langchain_core.documents import Document

documents = [Document(page_content = t) for t in chunk_ques_gen]


In [77]:
split_answer_gen = TokenTextSplitter(

    encoding_name='cl100k_base',
    chunk_size = 1000,
    chunk_overlap = 200
)

In [78]:
documents_answer_gen = split_answer_gen.split_documents(documents)
documents_answer_gen

[Document(metadata={}, page_content='Notes\n1\nNotes\nUnderstanding REST APIs\nREST \ue081Representational State Transfer) is an architectural style for designing \nnetworked applications. It relies on stateless, client-server communication over \nHTTP using standard methods and status codes. RESTful APIs are designed \naround resources, which can be anything from users and products to \ndocuments.\nKey Concepts:\n\ue072\ue094\x00 Resources: Everything that can be accessed via a RESTful API is \nconsidered a "resource." Each resource has a unique identifier \ue081URI\ue082.\n\ue073\ue094\x00 Representations: Resources are transferred in some representation like \nJSON or XML.\n\ue074\ue094\x00 Stateless Communication: Each request from client to server contains all \nneeded information; the server does not store any state about the client \nsession.\n\ue075\ue094\x00 HTTP Methods: REST APIs use standard HTTP methods to perform actions \non resources.\n\ue076\ue094\x00 HTTP Status Codes

In [79]:
from langchain_groq import ChatGroq
 
llm_question_gen_pipeline = ChatGroq(
    model='openai/gpt-oss-20b'
)


In [80]:
prompt = """
you are an expert at creating questions based on coding materials and documentation.
your goal is to prepare a coder or programmer for their exam and coding tests.
you do this by asking questions about the text below:


----------
{text}
----------

create a questions that will prepare the coders or programmers for their tests.
make sure not to lose any important information 

QUESTIONS:



"""

In [81]:
from langchain_core.prompts import  PromptTemplate

In [82]:
PROMPT_QUESTIONS = PromptTemplate(template=prompt,input_variables=['text'])


In [83]:
refine_template  = ("""

you are an expert at creating practice questions based on coding material and documentation
your goal is to help a coder or programmer prepare for coding test.
we have received some practice questions to certain extent: {existing_answer}
we have the option to refine the existing questions or add new ones.
(only if necessary ) with some more context below.
                    
----------
{text}
----------
                    
Given the new context, refine the original questions in english 
if the context is not helpful, please provide the original questions 
QUESTIONS: 
                    


""")

In [84]:
REFINE_PROMPT_QUESTIONS = PromptTemplate(

    input_variables=["existing_answer","text"],
    template=refine_template,
)

In [85]:
from langchain_core.output_parsers import StrOutputParser


In [86]:
initial_chain = PROMPT_QUESTIONS | llm_question_gen_pipeline | StrOutputParser()
refine_chain = REFINE_PROMPT_QUESTIONS | llm_question_gen_pipeline | StrOutputParser()



In [87]:
all_questions = []

for doc in documents_answer_gen:
    questions = initial_chain.invoke({"text":doc.page_content})
    all_questions.append(questions)

In [88]:
print('/nRefining and consolidating all questions')
final_questions = refine_chain.invoke({"existing_answer":all_questions,"text":"Consolidate these questions into a comprehensive exam preparation guide. Remove duplicates and organize by topic"})


/nRefining and consolidating all questions


In [89]:
with open("exam.txt","w") as file:
    file.write(final_questions)

In [91]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')


/opt/miniconda3/envs/interview/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10003.78it/s]
